In [1]:
import numpy as np
import random as rd
import copy
import psutil
import gc
import pickle
from qiskit.quantum_info import SparsePauliOp
n_qubit = 1
dim     = 2**n_qubit
nld     = 11
lds     = np.linspace(0,1,num=nld)

In [2]:
def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")

In [3]:
t_x          = 0.5
t_zs         = []
hamiltonians = []
alpha        = 0
for ld in lds:
    t_z = -1 + 2.0 * ld
    h = SparsePauliOp.from_list([('X',t_x),('Z',t_z)])
    t_zs.append(t_z)
    hamiltonians.append(h)
    print(hamiltonians[alpha])
    alpha += 1


SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -1. +0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.8+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.6+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.4+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.2+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0. +0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.2+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.4+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.6+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.8+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 1. +0.j])


In [4]:
hamiltonian_diffs = []
for ild in range(nld-1):
    hamiltonian_diffs.append((hamiltonians[ild+1]-hamiltonians[ild]).simplify())
    print(hamiltonian_diffs[ild])

SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])


In [5]:
hamiltonian_diffs_list = []
for ild in range(nld-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[ild].to_list())
    print(hamiltonian_diffs_list[ild])

[('Z', (0.19999999999999996+0j))]
[('Z', (0.20000000000000007+0j))]
[('Z', (0.20000000000000007+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.20000000000000018+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]


In [6]:
n_hamiltonians = len(hamiltonians)

In [7]:
# exact eigenvalues
eigen_energies_exact  = np.zeros((nld,dim),dtype=float)
eigen_vectors_exact   = np.zeros((nld,dim,dim),dtype=complex)
for ild in range(nld):
    eigen_e, eigen_v = np.linalg.eigh(hamiltonians[ild].to_matrix())
    indx = np.argsort(eigen_e.real)
    for i in range(dim):
        eigen_energies_exact[ild,i]   = eigen_e[indx[i]].real
        eigen_vectors_exact[ild,:,i] = eigen_v[:,indx[i]]


In [8]:
x_op = SparsePauliOp.from_list([('X',1.0)])
y_op = SparsePauliOp.from_list([('Y',1.0)])
z_op = SparsePauliOp.from_list([('Z',1.0)])
x_mat = x_op.to_matrix()
y_mat = y_op.to_matrix()
z_mat = z_op.to_matrix()

In [9]:
def ComputeUnitaryParams(U):
    theta = 2.0 * np.arccos(np.abs(U[0,0]))
    if (theta<1E-6):
        theta = 2.0*np.arcsin(np.abs(U[0,1]))
    gamma = np.angle(U[0,0])
    if (theta<1E-10):
        phi   = np.angle(U[1,1]/U[0,0])
        lam   = 0
    else:
        phi   = np.angle(U[1,0]/np.sin(theta/2)) - gamma
        lam   = np.angle(-U[0,1]/np.sin(theta/2)) - gamma
    return [theta, phi, lam, gamma]

def ExactTimeEvolution (alpha, time):
    Vl = copy.deepcopy(eigen_vectors_exact[alpha,:,:])
    evol = np.zeros((dim,dim),dtype=complex)
    exp_d = np.diag(np.exp(-1j*eigen_energies_exact[alpha,:]*time))
    evol = Vl@exp_d@Vl.conj().T
    return evol

def TrotterTimeEvolution (alpha, Nt, time):
    dtime  = time/Nt
    dtx = dtime * t_x
    dtz = dtime * t_zs[alpha]
    evol = np.identity(dim)
    for it in range(Nt):
        evol_X = np.cos(dtx)*np.identity(dim) - 1j*np.sin(dtx) * x_mat
        evol_Z = np.cos(dtz)*np.identity(dim) - 1j*np.sin(dtz) * z_mat
        evol = evol_Z@evol
        evol = evol_X@evol
    return evol

In [10]:
# exact results
norms_exact  = np.ones((nld,dim),dtype=float)
for i in range(dim):
    phi = eigen_vectors_exact[0,:,i]
    for alpha in range(1,nld):
        proj_matrix = np.outer(eigen_vectors_exact[alpha,:,i],eigen_vectors_exact[alpha,:,i].conj())
        phi = proj_matrix@phi
        norms_exact[alpha,i] = phi.conj()@phi

x_exps_exact = np.zeros((nld,dim),dtype=float)
y_exps_exact = np.zeros((nld,dim),dtype=float)
z_exps_exact = np.zeros((nld,dim),dtype=float)
for ild in range(nld):
    for i in range(dim):
        x_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@x_mat@eigen_vectors_exact[ild,:,i]
        y_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@y_mat@eigen_vectors_exact[ild,:,i]
        z_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@z_mat@eigen_vectors_exact[ild,:,i]

/tmp/ipykernel_9713/1490341378.py:8: ComplexWarning: Casting complex values to real discards the imaginary part
  norms_exact[alpha,i] = phi.conj()@phi
/tmp/ipykernel_9713/1490341378.py:15: ComplexWarning: Casting complex values to real discards the imaginary part
  x_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@x_mat@eigen_vectors_exact[ild,:,i]
/tmp/ipykernel_9713/1490341378.py:16: ComplexWarning: Casting complex values to real discards the imaginary part
  y_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@y_mat@eigen_vectors_exact[ild,:,i]
/tmp/ipykernel_9713/1490341378.py:17: ComplexWarning: Casting complex values to real discards the imaginary part
  z_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@z_mat@eigen_vectors_exact[ild,:,i]


In [11]:
nmc = int(400)


In [12]:
theta_init = np.zeros((dim),dtype=float)
for i in range(dim):
    theta_init[i] = 2.0 * np.angle(eigen_vectors_exact[0,0,i]+ 1j*eigen_vectors_exact[0,1,i])

In [13]:
def Circuit_Initialize(circuit_, i_, qr_):
    # initialize qr_[1:] to i_ th eigenvector
    circuit_.ry(theta_init[i_],qr_[1])

In [14]:
# observable amplitudes
n_obs = 3
# 5 different observables are computed
# 0; norm, 1; dE1, 2; Z
O_timelists         = [[[[] for _ in range(n_obs)] for _ in range(nld)] for _ in range(dim)]

In [15]:
beta = 5

In [16]:
with open('MDH.time.binary','rb') as file_:
    [O_timelists] = pickle.load(file_)

#for i in range(dim):
#    for alpha in range(1,nld):
#        for i_obs in range(n_obs):
#            for imc in range(nmc):
#                times = []
#                for alpha_ in range(1,alpha):
#                    time = rd.gauss(0.0, beta)
#                    times.append(time)
#
#                time = rd.gauss(0.0, beta)
#                times.append(time)
#
#                time = rd.gauss(0.0, beta)
#                times.append(time)
#
#                for alpha_ in reversed(range(1,alpha)):
#                    time = rd.gauss(0.0, beta)
#                    times.append(time)
#                O_timelists[i][alpha][i_obs].append(times)
#with open('MDH.time.binary','wb') as file_:
#    pickle.dump([O_timelists],file_)
#

In [17]:
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler, Estimator, Session, Options
from qiskit.providers import JobStatus
import time as time_lib

service = QiskitRuntimeService(channel="ibm_quantum")

def run_estimator (max_circuits, estimator, circuits, observables):
    len_circuits = len(circuits)
    num_jobs = len_circuits//max_circuits
    remainder = len_circuits%max_circuits
    if (remainder>0):
        num_jobs += 1
    i_start = 0
    job_ids = []
    for _ in range(num_jobs):
        i_end = min(i_start + max_circuits,len_circuits)
        job = estimator.run(circuits[i_start:i_end], observables[i_start:i_end])
        job_ids.append(job.job_id())
        i_start += max_circuits
    return job_ids

def check_jobs (service, max_circuits, estimator, circuits, observables, job_ids):
    len_circuits    = len(circuits)
    num_jobs        = len(job_ids)
    done            = True
    changed         = False
    i_start = 0
    for ind_job in range(num_jobs):
        job_id = job_ids[ind_job]
        job = service.job(job_id)
        if job.status() is JobStatus.DONE:
            i_start += max_circuits
            continue
        else:
            done = False
            # resubmit if there is a problem
            if job.status() in [JobStatus.ERROR, JobStatus.CANCELLED]:
                #print(job.status())
                #if (job.status() is JobStatus.ERROR):
                #    job.cancel()
                # no need to cancel..
                i_end = min(i_start + max_circuits,len_circuits)
                job_new = estimator.run(circuits[i_start:i_end], observables[i_start:i_end])
                job_ids[ind_job] = job_new.job_id()
                changed = True
                s = ''
                s += '## There was a problem in submitting job_ids[{ind_job}], '.format(ind_job=ind_job)
                s +='\n'
                s += '  {}'.format(job_id)
                s +='\n'
                s += 'The job is resubmitted. New job_ids[{ind_job}] is, '.format(ind_job=ind_job)
                s +='\n'
                s += '  {}'.format(job_ids[ind_job])
                print(s)

            i_start += max_circuits
    return [done, changed]


def moniter_jobs (service, max_circuits, estimator, circuits, observables, job_ids):
    len_circuits = len(circuits)
    num_jobs     = len(job_ids)
    done = False
    changed    = False
    while not done:
        done = True
        i_start = 0
        for ind_job in range(num_jobs):
            job_id = job_ids[ind_job]
            job = service.job(job_id)
            if job.status() is JobStatus.DONE:
                i_start += max_circuits
                continue
            else:
                done = False
                # resubmit if there is a problem
                if job.status() in [JobStatus.ERROR, JobStatus.CANCELLED]:
                    #print(job.status())
                    #if (job.status() is JobStatus.ERROR):
                    #    job.cancel()
                    # no need to cancel..
                    i_end = min(i_start + max_circuits,len_circuits)
                    job_new = estimator.run(circuits[i_start:i_end], observables[i_start:i_end])
                    job_ids[ind_job] = job_new.job_id()
                    changed = True
                    s = ''
                    s += '## There was a problem in submitting job_ids[{ind_job}], '.format(ind_job=ind_job)
                    s +='\n'
                    s += '  {}'.format(job_id)
                    s +='\n'
                    s += 'The job is resubmitted. New job_ids[{ind_job}] is, '.format(ind_job=ind_job)
                    s +='\n'
                    s += '  {}'.format(job_ids[ind_job])
                    print(s)

                i_start += max_circuits
        time_lib.sleep(10)
    return [done, changed]

def accumulate_job_results (service, job_ids):
    results = np.array([],dtype=float)
    for job_id in job_ids:
        job = service.job(job_id)
        while job.status() is not JobStatus.DONE:
            time_lib.sleep(5) 
        results = np.append(results,job.result().values)
    return results

In [18]:
backend_name = "ibmq_lima" # 
backend = service.get_backend(backend_name)

In [19]:
from qiskit import transpile, Aer
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.primitives import Sampler as AerSampler

from qiskit_aer.noise import NoiseModel

In [20]:
from qiskit import transpile, Aer
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.primitives import Sampler as AerSampler

from qiskit_aer.noise import NoiseModel

#noise_model = NoiseModel.from_backend(backend, readout_error=False)
noise_model = NoiseModel.from_backend(backend)

coupling_map = backend.coupling_map
basis_gates = backend.basis_gates
estimator = AerEstimator(backend_options={
        "method": "density_matrix",
        "coupling_map": coupling_map,
        "noise_model": noise_model,
        "basis_gates": basis_gates
    })

sampler = AerSampler(backend_options={
        "method": "density_matrix",
        "coupling_map": coupling_map,
        "noise_model": noise_model,
        "basis_gates": basis_gates
    })

num_qubits_backend = backend.num_qubits

nshot    = 4000

In [21]:
print(noise_model)

NoiseModel:
  Basis gates: ['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']
  Instructions with noise: ['sx', 'cx', 'x', 'reset', 'measure', 'id']
  Qubits with noise: [0, 1, 2, 3, 4]
  Specific qubit errors: [('id', (0,)), ('id', (1,)), ('id', (2,)), ('id', (3,)), ('id', (4,)), ('sx', (0,)), ('sx', (1,)), ('sx', (2,)), ('sx', (3,)), ('sx', (4,)), ('x', (0,)), ('x', (1,)), ('x', (2,)), ('x', (3,)), ('x', (4,)), ('cx', (4, 3)), ('cx', (3, 4)), ('cx', (0, 1)), ('cx', (1, 0)), ('cx', (3, 1)), ('cx', (1, 3)), ('cx', (2, 1)), ('cx', (1, 2)), ('reset', (0,)), ('reset', (1,)), ('reset', (2,)), ('reset', (3,)), ('reset', (4,)), ('measure', (0,)), ('measure', (1,)), ('measure', (2,)), ('measure', (3,)), ('measure', (4,))]


In [22]:
q_layout = [0,1]
z_hadamard_test = SparsePauliOp.from_sparse_list([('Z',[q_layout[0]],1)],num_qubits=num_qubits_backend)
print(z_hadamard_test)

SparsePauliOp(['IIIIZ'],
              coeffs=[1.+0.j])


In [23]:
from qiskit import QuantumRegister
from qiskit import QuantumCircuit, transpile

qr = QuantumRegister(n_qubit+1, 'q')

In [24]:
# parametrized circuit
from qiskit.circuit import ParameterVector
params = ParameterVector('θ',5)

In [25]:
# run qzmc;

norms_qzmc               = np.ones((n_hamiltonians,dim),dtype=float)
eigen_energies_qzmc      = np.zeros((n_hamiltonians,dim),dtype=float)
eigen_energies_qzmc[0,:] = eigen_energies_exact[0,:]

In [26]:
result_values_save = [[[] for _ in range(n_hamiltonians)] for _ in range(dim)]

In [27]:
#i = 0 # only consider the ground state
## real part measurement
#circuit_real = QuantumCircuit(qr,name='-')
#Circuit_Initialize(circuit_real,i,qr)
#circuit_real.h(qr[0])
#circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
#circuit_real.p(params[4],qr[0])
#circuit_real.h(qr[0])
#
#circuit_real = transpile(circuit_real,backend)
#
#eps = eigen_vectors_exact[0,:,i].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i]
#eps = eps.real
#for alpha in range(1,n_hamiltonians):
#    params_list = []
#    # 0; norm measurement
#    i_obs = 0
#    for imc in range(nmc):
#        U_evol = np.identity(dim,dtype=complex)
#        times  = O_timelists[i][alpha][i_obs][imc]
#        phase  = 0.0
#        i_time = 0
#        # construction of P
#        for alpha_ in range(1,alpha):
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#            phase += eigen_energies_qzmc[alpha_,i] * time
#            i_time += 1
#    
#        time = times[i_time]
#        U_evol = ExactTimeEvolution(alpha,time)@U_evol
#        phase += eps * time
#        i_time += 1
#    
#        # construction of P^\dagger
#    
#        time = times[i_time]
#        U_evol = ExactTimeEvolution(alpha,time)@U_evol
#        phase += eps * time
#        i_time += 1
#    
#        for alpha_ in reversed(range(1,alpha)):
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#            phase += eigen_energies_qzmc[alpha_,i] * time
#            i_time += 1
#
#        theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#        params_list.append([theta,phi,lam,gamma,phase])
#    # 1; dE1 measurement
#    i_obs = 1
#    nhd1 = len(hamiltonian_diffs[alpha-1])
#    for ihd in range(nhd1):
#        # pass for constant contribution
#        if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#            continue
#        for imc in range(nmc):
#            U_evol = np.identity(dim,dtype=complex)
#            times  = O_timelists[i][alpha][i_obs][imc]
#            phase  = 0.0
#            i_time = 0
#            # construction of P
#            for alpha_ in range(1,alpha):
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                phase += eigen_energies_qzmc[alpha_,i] * time
#                i_time += 1
#
#            # insertion of (H[alpha]-H[alpha-1])
#            Mat = hamiltonian_diffs[alpha-1].paulis[ihd].to_matrix()
#
#            U_evol = Mat@U_evol
#
#            # construction of P; continued
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha,time)@U_evol
#            phase += eps * time
#            i_time += 1
#
#            # construction of P^\dagger
#    
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha,time)@U_evol
#            phase += eps * time
#            i_time += 1
#    
#            for alpha_ in reversed(range(1,alpha)):
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                phase += eigen_energies_qzmc[alpha_,i] * time
#                i_time += 1
#            theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#            params_list.append([theta,phi,lam,gamma,phase])
#            
#    # 1; dE2 measurement
#    i_obs = 2
#    if (alpha<n_hamiltonians-1):
#        nhd2 = len(hamiltonian_diffs[alpha])
#    else:
#        nhd2 = 0
#    for ihd in range(nhd2):
#        # pass for constant contribution
#        if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#            continue
#        for imc in range(nmc):
#            U_evol = np.identity(dim,dtype=complex)
#            times  = O_timelists[i][alpha][i_obs][imc]
#            phase  = 0.0
#            i_time = 0
#            # construction of P
#            for alpha_ in range(1,alpha):
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                phase += eigen_energies_qzmc[alpha_,i] * time
#                i_time += 1
#
#            # construction of P; continued
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha,time)@U_evol
#            phase += eps * time
#            i_time += 1
#
#            # insertion of (H[alpha+1]-H[alpha])
#            Mat = hamiltonian_diffs[alpha].paulis[ihd].to_matrix()
#
#            U_evol = Mat@U_evol
#
#            # construction of P^\dagger
#    
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha,time)@U_evol
#            phase += eps * time
#            i_time += 1
#    
#            for alpha_ in reversed(range(1,alpha)):
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                phase += eigen_energies_qzmc[alpha_,i] * time
#                i_time += 1
#            theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#            params_list.append([theta,phi,lam,gamma,phase])
#    circuits = [circuit_real.assign_parameters({params: params_}) for params_ in params_list]
#
#    job = estimator.run(circuits,[z_hadamard_test]*len(circuits))
#    result_values = job.result().values
#    result_values_save[i][alpha] = result_values
#    # compute values
#    norm    = 0.0
#    dE1     = 0.0
#    dE2     = 0.0
#
#    i_meas = 0
#    # 0; norm
#    for imc in range(nmc):
#        norm += result_values_save[i][alpha][i_meas]
#        i_meas += 1
#    # 1; dE1
#    nhd1 = len(hamiltonian_diffs[alpha-1])
#    for ihd in range(nhd1):
#        if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#            continue
#        coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
#        for imc in range(nmc):
#            dE1 +=  coeff * result_values_save[i][alpha][i_meas]
#            i_meas += 1
#
#    # 2; dE2
#    if (alpha<n_hamiltonians-1):
#        nhd2 = len(hamiltonian_diffs[alpha])
#    else:
#        nhd2 = 0
#    for ihd in range(nhd2):
#        if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#            continue
#        coeff = hamiltonian_diffs_list[alpha][ihd][1]
#        for imc in range(nmc):
#            dE2 += coeff * result_values_save[i][alpha][i_meas]
#            i_meas += 1
#    
#    dE1     /=norm
#    dE2     /=norm
#
#    norm    /=nmc
#
#    # add constant contributions
#    for ihd in range(nhd1):
#        if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#            dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]
#
#    for ihd in range(nhd2):
#        if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#            dE2 += hamiltonian_diffs_list[alpha][ihd][1]
#
#    eigen_energies_qzmc[alpha,i] = eigen_energies_qzmc[alpha-1,i] + dE1
#    norms_qzmc[alpha,i] = norm
#
#    if (alpha<n_hamiltonians-1):
#        eps = eigen_energies_qzmc[alpha,i] + dE2
#        eps = eps.real
#        
#    print(alpha, norms_qzmc[alpha,i], eigen_energies_qzmc[alpha,i]-eigen_energies_exact[alpha,i])
#    if (alpha<n_hamiltonians-1):
#        print(alpha, eps, eigen_energies_exact[alpha+1,i])
#    st = '# {i}/{dim}: {percent}%'.format(i=i+1,dim=dim,percent=((alpha)/(n_hamiltonians-1)*100))
#    print(st)
#            
#
#

In [28]:
rep_list = [0, 1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
n_rep = len(rep_list)
print(n_rep)

15


In [29]:
# run qzmc;

norms_rep                = np.ones((n_rep,n_hamiltonians),dtype=float)
eigen_energies_rep       = np.zeros((n_rep,n_hamiltonians),dtype=float)
eigen_energies_rep[:,0]  = eigen_energies_exact[0,0]
Z_rep                    = np.zeros((n_rep),dtype=float)

In [30]:
result_values_rep   = [[[] for _ in range(n_hamiltonians)] for _ in range(n_rep)]
result_values_Z_rep = [[] for _ in range(n_rep)]

In [31]:
memory_usage('before run')

[before run] memory usage:    0.24432 GiB


In [32]:
i = 0 # only consider the ground state
for i_rep in range(n_rep):
#for i_rep in range(1):
    rep = rep_list[i_rep]
    # real part measurement
    circuit_real = QuantumCircuit(qr,name='-')
    Circuit_Initialize(circuit_real,i,qr)
    circuit_real.h(qr[0])
    for r in range(rep):
        circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
        circuit_real.cu(-params[0],-params[2],-params[1],-params[3],qr[0],qr[1])
    circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
    circuit_real.p(params[4],qr[0])
    circuit_real.h(qr[0])

    circuit_real = transpile(circuit_real,backend)

    eps = eigen_vectors_exact[0,:,i].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i]
    eps = eps.real
    for alpha in range(1,n_hamiltonians):
        params_list = []
        # 0; norm measurement
        i_obs = 0
        for imc in range(nmc):
            U_evol = np.identity(dim,dtype=complex)
            times  = O_timelists[i][alpha][i_obs][imc]
            phase  = 0.0
            i_time = 0
            # construction of P
            for alpha_ in range(1,alpha):
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                phase += eigen_energies_rep[i_rep,alpha_] * time
                i_time += 1

            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha,time)@U_evol
            phase += eps * time
            i_time += 1

            # construction of P^\dagger

            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha,time)@U_evol
            phase += eps * time
            i_time += 1

            for alpha_ in reversed(range(1,alpha)):
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                phase += eigen_energies_rep[i_rep,alpha_] * time
                i_time += 1

            theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
            params_list.append([theta,phi,lam,gamma,phase])
        # 1; dE1 measurement
        i_obs = 1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            # pass for constant contribution
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[i][alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_rep[i_rep,alpha_] * time
                    i_time += 1

                # insertion of (H[alpha]-H[alpha-1])
                Mat = hamiltonian_diffs[alpha-1].paulis[ihd].to_matrix()

                U_evol = Mat@U_evol

                # construction of P; continued
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                # construction of P^\dagger

                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_rep[i_rep,alpha_] * time
                    i_time += 1
                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])

        # 2; dE2 measurement
        i_obs = 2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            # pass for constant contribution
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[i][alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_rep[i_rep,alpha_] * time
                    i_time += 1

                # construction of P; continued
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                # insertion of (H[alpha+1]-H[alpha])
                Mat = hamiltonian_diffs[alpha].paulis[ihd].to_matrix()

                U_evol = Mat@U_evol

                # construction of P^\dagger

                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_rep[i_rep,alpha_] * time
                    i_time += 1
                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])
        # 3; Z measurement
        if (alpha==n_hamiltonians-1):
            i_obs = 2
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[i][alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_rep[i_rep,alpha_] * time
                    i_time += 1

                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                # insertion of Z

                U_evol = z_mat@U_evol

                # construction of P^\dagger

                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_rep[i_rep,alpha_] * time
                    i_time += 1

                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])

        circuits = [circuit_real.assign_parameters({params: params_}) for params_ in params_list]

        z_hadamards = [z_hadamard_test]*len(circuits)
        #estimator = AerEstimator(run_options={"shots": 4000})
        estimator = AerEstimator(backend_options={
                "method": "density_matrix",
                "coupling_map": coupling_map,
                "noise_model": noise_model,
                "basis_gates": basis_gates
            },run_options={"shots": 4000})
        job = estimator.run(circuits,z_hadamards)
        result_values = job.result().values
        result_values_rep[i_rep][alpha] = result_values
        # compute values
        norm    = 0.0
        dE1     = 0.0
        dE2     = 0.0

        i_meas = 0
        # 0; norm
        for imc in range(nmc):
            norm += result_values_rep[i_rep][alpha][i_meas]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                dE1 +=  coeff * result_values_rep[i_rep][alpha][i_meas]
                i_meas += 1

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                dE2 += coeff * result_values_rep[i_rep][alpha][i_meas]
                i_meas += 1

        dE1     /=norm
        dE2     /=norm

        # 3; Z
        if (alpha==n_hamiltonians-1):
            Znorm = 0.0
            for imc in range(nmc):
                Znorm += result_values_rep[i_rep][alpha][i_meas]
                i_meas += 1
            Znorm /= norm

        norm    /=nmc

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2 += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_rep[i_rep,alpha] = eigen_energies_rep[i_rep,alpha-1] + dE1
        norms_rep[i_rep,alpha] = norm

        if (alpha<n_hamiltonians-1):
            eps = eigen_energies_rep[i_rep,alpha] + dE2
            eps = eps.real
        else:
            Z_rep[i_rep] = Znorm

        print(alpha, norms_rep[i_rep,alpha], eigen_energies_rep[i_rep,alpha]-eigen_energies_exact[alpha,i])
        if (alpha<n_hamiltonians-1):
            print(alpha, eps, eigen_energies_exact[alpha+1,i])
        else:
            print('# Zdiff: ', Znorm-z_exps_exact[-1,i])
        st = '# {i_rep}/{n_rep}: {percent}%'.format(i_rep=i_rep+1,n_rep=n_rep,percent=((alpha)/(n_hamiltonians-1)*100))

        del job
        del z_hadamards
        del circuits 
        del estimator
        collected = gc.collect()
        memory_usage(st)




/tmp/ipykernel_9713/819386927.py:256: ComplexWarning: Casting complex values to real discards the imaginary part
  eigen_energies_rep[i_rep,alpha] = eigen_energies_rep[i_rep,alpha-1] + dE1


1 0.9265737500000002 0.00034129168071661553
1 -0.7733706524531347 -0.7810249675906655
[# 1/15: 10.0%] memory usage:    0.32931 GiB
2 0.9220575000000006 -2.406157859546898e-05
2 -0.6274327959083201 -0.6403124237432848
[# 1/15: 20.0%] memory usage:    0.34051 GiB
3 0.9100162500000003 -1.5491929376665325e-05
3 -0.5165441370862901 -0.5385164807134504
[# 1/15: 30.0%] memory usage:    0.34253 GiB
4 0.8838100000000004 0.0028555749184774326
4 -0.4620231895437425 -0.5
[# 1/15: 40.0%] memory usage:    0.34188 GiB
5 0.8283999999999999 0.0022081791881269908
5 -0.49701592752360646 -0.5385164807134505
[# 1/15: 50.0%] memory usage:    0.34188 GiB
6 0.8059100000000002 0.0010201643623732881
6 -0.6141798169901063 -0.640312423743285
[# 1/15: 60.0%] memory usage:    0.34283 GiB
7 0.8054450000000002 -0.0072879344728481454
7 -0.7764946340512304 -0.7810249675906655
[# 1/15: 70.0%] memory usage:    0.34279 GiB
8 0.7993537500000011 -0.008262659872656175
8 -0.9457527719628627 -0.9433981132056604
[# 1/15: 80.0%]

In [33]:
#with open('MDH.rep.results','rb') as file_:
#    [result_values_rep] = pickle.load(file_)

with open('MDH.rep.results','wb') as file_:
    pickle.dump([result_values_rep],file_)


In [34]:
norms_raw_rep            = np.zeros((n_rep,n_hamiltonians,nmc),dtype=float)
dEnorm_raw_rep           = np.zeros((n_rep,n_hamiltonians,nmc),dtype=float)

In [35]:
i = 0 # only consider the ground state
for i_rep in range(n_rep):
    rep = rep_list[i_rep]
    eps = eigen_vectors_exact[0,:,i].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i]
    eps = eps.real
    for alpha in range(1,n_hamiltonians):
        # compute values
        norm    = 0.0
        dE1     = 0.0
        dE2     = 0.0

        i_meas = 0
        # 0; norm
        for imc in range(nmc):
            norms_raw_rep[i_rep][alpha][imc] = result_values_rep[i_rep][alpha][i_meas]
            norm += norms_raw_rep[i_rep][alpha][imc]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                dEnorm_raw_rep[i_rep][alpha][imc] +=  coeff * result_values_rep[i_rep][alpha][i_meas]
                i_meas += 1
        for imc in range(nmc):
            dE1 += dEnorm_raw_rep[i_rep][alpha][imc]

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                dE2 += coeff * result_values_rep[i_rep][alpha][i_meas]
                i_meas += 1

        dE1     /=norm
        dE2     /=norm
        norm    /=nmc

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2 += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_rep[i_rep,alpha] = eigen_energies_rep[i_rep,alpha-1] + dE1
        norms_rep[i_rep,alpha] = norm

        if (alpha<n_hamiltonians-1):
            eps = eigen_energies_rep[i_rep,alpha] + dE2
            eps = eps.real

        print(alpha, norms_rep[i_rep,alpha], eigen_energies_rep[i_rep,alpha]-eigen_energies_exact[alpha,i])
        if (alpha<n_hamiltonians-1):
            print(alpha, eps, eigen_energies_exact[alpha+1,i])
        st = '# {i_rep}/{n_rep}: {percent}%'.format(i_rep=i_rep+1,n_rep=n_rep,percent=((alpha)/(n_hamiltonians-1)*100))

        #collected = gc.collect()
        memory_usage(st)




1 0.9265737500000002 0.00034129168071661553
1 -0.7733706524531347 -0.7810249675906655
[# 1/15: 10.0%] memory usage:    1.80170 GiB
2 0.9220575000000006 -2.406157859546898e-05
2 -0.6274327959083201 -0.6403124237432848
[# 1/15: 20.0%] memory usage:    1.80170 GiB
3 0.9100162500000003 -1.5491929376665325e-05
3 -0.5165441370862901 -0.5385164807134504
[# 1/15: 30.0%] memory usage:    1.80170 GiB
4 0.8838100000000004 0.0028555749184774326
4 -0.4620231895437425 -0.5
[# 1/15: 40.0%] memory usage:    1.80170 GiB
5 0.8283999999999999 0.0022081791881269908
5 -0.49701592752360646 -0.5385164807134505
[# 1/15: 50.0%] memory usage:    1.80170 GiB
6 0.8059100000000002 0.0010201643623732881
6 -0.6141798169901063 -0.640312423743285
[# 1/15: 60.0%] memory usage:    1.80170 GiB
7 0.8054450000000002 -0.0072879344728481454
7 -0.7764946340512304 -0.7810249675906655
[# 1/15: 70.0%] memory usage:    1.80170 GiB
8 0.7993537500000011 -0.008262659872656175
8 -0.9457527719628627 -0.9433981132056604
[# 1/15: 80.0%]

/tmp/ipykernel_9713/3772743504.py:25: ComplexWarning: Casting complex values to real discards the imaginary part
  dEnorm_raw_rep[i_rep][alpha][imc] +=  coeff * result_values_rep[i_rep][alpha][i_meas]


In [39]:
result_values_Z_rep = [[] for _ in range(n_rep)]

In [40]:
# Z measurements
i = 0 # only consider the ground state
#for i_rep in range(n_rep-1,n_rep):
for i_rep in range(n_rep):
    rep = rep_list[i_rep]
    # real part measurement
    circuit_real = QuantumCircuit(qr,name='-')
    Circuit_Initialize(circuit_real,i,qr)
    circuit_real.h(qr[0])
    for r in range(rep):
        circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
        circuit_real.cu(-params[0],-params[2],-params[1],-params[3],qr[0],qr[1])
    circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
    circuit_real.p(params[4],qr[0])
    circuit_real.h(qr[0])

    circuit_real = transpile(circuit_real,backend)

    params_list = []
    alpha = n_hamiltonians-1
    # 0; norm measurement
    i_obs = 0
    for imc in range(nmc):
        U_evol = np.identity(dim,dtype=complex)
        times  = O_timelists[i][alpha][i_obs][imc]
        phase  = 0.0
        i_time = 0
        # construction of P
        for alpha_ in range(1,alpha):
            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha_,time)@U_evol
            phase += eigen_energies_rep[i_rep,alpha_] * time
            i_time += 1

        time = times[i_time]
        U_evol = ExactTimeEvolution(alpha,time)@U_evol
        phase += eigen_energies_rep[i_rep,alpha] * time
        i_time += 1

        # construction of P^\dagger

        time = times[i_time]
        U_evol = ExactTimeEvolution(alpha,time)@U_evol
        phase += eigen_energies_rep[i_rep,alpha] * time
        i_time += 1

        for alpha_ in reversed(range(1,alpha)):
            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha_,time)@U_evol
            phase += eigen_energies_rep[i_rep,alpha_] * time
            i_time += 1

        theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
        params_list.append([theta,phi,lam,gamma,phase])
    # 3; Z measurement
    i_obs = 2
    for imc in range(nmc):
        U_evol = np.identity(dim,dtype=complex)
        times  = O_timelists[i][alpha][i_obs][imc]
        phase  = 0.0
        i_time = 0
        # construction of P
        for alpha_ in range(1,alpha):
            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha_,time)@U_evol
            phase += eigen_energies_rep[i_rep,alpha_] * time
            i_time += 1

        time = times[i_time]
        U_evol = ExactTimeEvolution(alpha,time)@U_evol
        phase += eigen_energies_rep[i_rep,alpha] * time
        i_time += 1

        # insertion of Z

        U_evol = z_mat@U_evol

        # construction of P^\dagger

        time = times[i_time]
        U_evol = ExactTimeEvolution(alpha,time)@U_evol
        phase += eigen_energies_rep[i_rep,alpha] * time
        i_time += 1

        for alpha_ in reversed(range(1,alpha)):
            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha_,time)@U_evol
            phase += eigen_energies_rep[i_rep,alpha_] * time
            i_time += 1

        theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
        params_list.append([theta,phi,lam,gamma,phase])

    circuits = [circuit_real.assign_parameters({params: params_}) for params_ in params_list]

    z_hadamards = [z_hadamard_test]*len(circuits)
    #estimator = AerEstimator(run_options={"shots": 4000})
    estimator = AerEstimator(backend_options={
            "method": "density_matrix",
            "coupling_map": coupling_map,
            "noise_model": noise_model,
            "basis_gates": basis_gates
        },run_options={"shots": 4000})
    job = estimator.run(circuits,z_hadamards)
    result_values = job.result().values
    result_values_Z_rep[i_rep] = result_values
    # compute values
    norm    = 0.0

    i_meas = 0
    # 0; norm
    for imc in range(nmc):
        norm += result_values_Z_rep[i_rep][i_meas]
        i_meas += 1

    # 3; Z
    Znorm = 0.0
    for imc in range(nmc):
        Znorm += result_values_Z_rep[i_rep][i_meas]
        i_meas += 1
    Znorm /= norm
    norm  /=nmc

    Z_rep[i_rep] = Znorm

    print('# Zdiff: ', Znorm-z_exps_exact[-1,i])
    st = '# {i_rep}/{n_rep}: {percent}%'.format(i_rep=i_rep+1,n_rep=n_rep,percent=((alpha)/(n_hamiltonians-1)*100))

    del job
    del z_hadamards
    del circuits 
    del estimator
    collected = gc.collect()
    memory_usage(st)





# Zdiff:  0.0028157720022194965
[# 1/15: 100.0%] memory usage:    1.72668 GiB
# Zdiff:  0.002624928992179809
[# 2/15: 100.0%] memory usage:    1.72376 GiB
# Zdiff:  0.004680269882572752
[# 3/15: 100.0%] memory usage:    1.72278 GiB
# Zdiff:  0.0044077888142436406
[# 4/15: 100.0%] memory usage:    1.72278 GiB
# Zdiff:  0.003851440667762329
[# 5/15: 100.0%] memory usage:    1.72278 GiB
# Zdiff:  0.004316259897835906
[# 6/15: 100.0%] memory usage:    1.72278 GiB
# Zdiff:  0.004786682146274535
[# 7/15: 100.0%] memory usage:    1.72295 GiB
# Zdiff:  0.005369890143777867
[# 8/15: 100.0%] memory usage:    1.74341 GiB
# Zdiff:  0.005223209072869106
[# 9/15: 100.0%] memory usage:    1.76281 GiB
# Zdiff:  0.007457460955530015
[# 10/15: 100.0%] memory usage:    1.77465 GiB
# Zdiff:  0.008252703382853999
[# 11/15: 100.0%] memory usage:    1.79348 GiB
# Zdiff:  0.008099814401874239
[# 12/15: 100.0%] memory usage:    1.79735 GiB
# Zdiff:  0.0022483909791016776
[# 13/15: 100.0%] memory usage:    1.76

In [41]:
#with open('MDH.rep.Z.results','rb') as file_:
#    [result_values_Z_rep] = pickle.load(file_)

with open('MDH.rep.Z.results','wb') as file_:
    pickle.dump([result_values_Z_rep],file_)


In [42]:
Znorm_raw_rep            = np.zeros((n_rep,nmc),dtype=float)
norms_raw_rep_z          = np.zeros((n_rep,nmc),dtype=float)

In [43]:
norm_rep_z = np.zeros((n_rep),dtype=float)

In [44]:
i = 0 # only consider the ground state
for i_rep in range(n_rep):
    rep = rep_list[i_rep]

    alpha = n_hamiltonians-1
    # compute values
    norm    = 0.0
    Znorm   = 0.0
    i_meas = 0
    # 0; norm
    for imc in range(nmc):
        norms_raw_rep_z[i_rep][imc] =  result_values_Z_rep[i_rep][i_meas]
        i_meas += 1

    # 3; Z
    for imc in range(nmc):
        Znorm_raw_rep[i_rep][imc] = result_values_Z_rep[i_rep][i_meas]
        i_meas += 1
    
    for imc in range(nmc):
        norm += norms_raw_rep_z[i_rep][imc]
        Znorm += Znorm_raw_rep[i_rep][imc]
    Znorm /= norm
    norm  /=nmc

    Z_rep[i_rep] = Znorm
    norm_rep_z[i_rep] = norm

    print('# norm: ', norm)
    print('# Zdiff: ', Znorm-z_exps_exact[-1,i])
    st = '# {i_rep}/{n_rep}: {percent}%'.format(i_rep=i_rep+1,n_rep=n_rep,percent=((alpha)/(n_hamiltonians-1)*100))

    memory_usage(st)





# norm:  0.7998075000000004
# Zdiff:  0.0028157720022194965
[# 1/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.7659125
# Zdiff:  0.002624928992179809
[# 2/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.7350475000000002
# Zdiff:  0.004680269882572752
[# 3/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.7048175000000002
# Zdiff:  0.0044077888142436406
[# 4/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.6745762499999999
# Zdiff:  0.003851440667762329
[# 5/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.6465725000000007
# Zdiff:  0.004316259897835906
[# 6/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.5214662500000005
# Zdiff:  0.004786682146274535
[# 7/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.42224500000000004
# Zdiff:  0.005369890143777867
[# 8/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.3402875
# Zdiff:  0.005223209072869106
[# 9/15: 100.0%] memory usage:    1.77903 GiB
# norm:  0.2748599999999999
# Zdiff:  0.007457460955530015
[# 10/15: 1

In [45]:
# compute standard deviations
import random as rd
std_norm    = np.zeros((n_rep,n_hamiltonians),dtype=float)
std_dE      = np.zeros((n_rep,n_hamiltonians),dtype=float)
std_E       = np.zeros((n_rep,n_hamiltonians),dtype=float)
n_boot = 1000
for i_rep in range(n_rep):
    for alpha in range(1,n_hamiltonians):
        norm_boot = np.zeros((n_boot),dtype=float)
        dE_boot   = np.zeros((n_boot),dtype=float)
        for i_boot in range(n_boot):
            norm_ = 0.0
            dE_   = 0.0
            for imc in range(nmc):
                jmc = rd.randrange(nmc)
                norm_ += norms_raw_rep[i_rep,alpha,jmc]
                dE_ += dEnorm_raw_rep[i_rep,alpha,jmc]

            dE_   = dE_/norm_
            norm_ = norm_/nmc
            norm_boot[i_boot] = norm_
            dE_boot[i_boot] = dE_
        norm_boot_mean = 0.0
        dE_boot_mean   = 0.0
        for i_boot in range(n_boot):
            norm_boot_mean += norm_boot[i_boot]
            dE_boot_mean   += dE_boot[i_boot]
        norm_boot_mean /= n_boot
        dE_boot_mean /= n_boot

        var_norm = 0.0
        var_dE = 0.0
        for i_boot in range(n_boot):
            var_norm += (norm_boot[i_boot]-norm_boot_mean)**2
            var_dE += (dE_boot[i_boot]-dE_boot_mean)**2
        var_norm /= (n_boot-1)
        var_dE    /= (n_boot-1)
        std_norm[i_rep,alpha] = np.sqrt(var_norm)
        std_dE[i_rep,alpha] = np.sqrt(var_dE)
        std_E[i_rep,alpha] = std_E[i_rep,alpha-1] + var_dE
for i_rep in range(n_rep):
    for alpha in range(n_hamiltonians):
        std_E[i_rep,alpha] = np.sqrt(std_E[i_rep,alpha])

In [46]:
# compute standard deviations
import random as rd
std_norm_z  = np.zeros((n_rep),dtype=float)
std_Z       = np.zeros((n_rep),dtype=float)
std_Znorm   = np.zeros((n_rep),dtype=float)
n_boot = 1000
for i_rep in range(n_rep):
    norm_boot  = np.zeros((n_boot),dtype=float)
    Znorm_boot = np.zeros((n_boot),dtype=float)
    Z_boot     = np.zeros((n_boot),dtype=float)
    for i_boot in range(n_boot):
        norm_ = 0.0
        Z_ = 0.0
        for imc in range(nmc):
            jmc = rd.randrange(nmc)
            norm_ += norms_raw_rep_z[i_rep,jmc]
            Z_ += Znorm_raw_rep[i_rep,jmc]
        Znorm_boot[i_boot] = Z_/nmc
        Z_ = Z_/norm_
        norm_ = norm_/nmc
        norm_boot[i_boot] = norm_
        Z_boot[i_boot] = Z_
    norm_boot_mean = 0.0
    Znorm_boot_mean = 0.0
    Z_boot_mean = 0.0
    for i_boot in range(n_boot):
        norm_boot_mean += norm_boot[i_boot]
        Znorm_boot_mean += Znorm_boot[i_boot]
        Z_boot_mean += Z_boot[i_boot]
    norm_boot_mean /= n_boot
    Znorm_boot_mean /= n_boot
    Z_boot_mean /= n_boot

    var_norm = 0.0
    for i_boot in range(n_boot):
        var_norm += (norm_boot[i_boot]-norm_boot_mean)**2
    var_norm /= (n_boot-1)
    std_norm_z[i_rep] = np.sqrt(var_norm)

    var_Z = 0.0
    for i_boot in range(n_boot):
        var_Z += (Z_boot[i_boot]-Z_boot_mean)**2
    var_Z    /= (n_boot-1)
    std_Z[i_rep] = np.sqrt(var_Z)

    var_Znorm = 0.0
    for i_boot in range(n_boot):
        var_Znorm += (Znorm_boot[i_boot]-Znorm_boot_mean)**2
    var_Znorm    /= (n_boot-1)
    std_Znorm[i_rep] = np.sqrt(var_Znorm)

for i_rep in range(n_rep):
    print(std_norm_z[i_rep],std_Z[i_rep],std_Znorm[i_rep])

0.005751364844305763 0.012614402142041691 0.008413452467732944
0.005439745188247602 0.012325446525351304 0.00786979134951238
0.0052806082070531585 0.012902774240611996 0.007642289445314448
0.005083399660550398 0.01205056417507656 0.007053210466513354
0.004862419432474561 0.012851402499195735 0.00723528112236098
0.004575795870218318 0.012609487455627292 0.006820325713150256
0.003750515472141874 0.013233874276734411 0.005608587327787068
0.0030411312224328108 0.012410492138096639 0.004523612494013444
0.002481076444990523 0.013232662356040781 0.0038399901199035564
0.0019842349469126725 0.013428429092277469 0.0031229274068072972
0.0016835862382100484 0.013070015166110432 0.002457562505621736
0.0014703667635217015 0.0134972560476402 0.0020405399801483952
0.0013013145758497629 0.01440424013316709 0.0017574992535207862
0.0011301045738819288 0.015926163785044068 0.0014975396129370837
0.0010483592359552174 0.017583752205290636 0.0013513641245591736


In [47]:
for i_rep in range(n_rep):
    print(norm_rep_z[i_rep], Z_rep[i_rep], norm_rep_z[i_rep]*Z_rep[i_rep])

0.7998075000000004 -0.8916114189976962 -0.7131175000000003
0.7659125 -0.8918022620077359 -0.6830425
0.7350475000000002 -0.889746921117343 -0.6540062500000003
0.7048175000000002 -0.8900194021856721 -0.6273012500000001
0.6745762499999999 -0.8905757503321534 -0.6007612500000002
0.6465725000000007 -0.8901109311020798 -0.5755212500000001
0.5214662500000005 -0.8896405088536412 -0.4639175000000005
0.42224500000000004 -0.8890573008561379 -0.37539999999999996
0.3402875 -0.8892039819270466 -0.3025849999999999
0.2748599999999999 -0.8869697300443857 -0.24379249999999975
0.22303875000000026 -0.8861744876170617 -0.19765125000000017
0.1794187500000002 -0.8863273765980415 -0.15902375000000005
0.1441275000000001 -0.8921788000208141 -0.12858749999999997
0.11629750000000001 -0.8758679249338978 -0.10186124999999999
0.09546750000000002 -0.8768429046534162 -0.08371000000000003


In [48]:
for i_rep in range(n_rep):
    print(std_norm_z[i_rep],std_Z[i_rep],std_Znorm[i_rep])

0.005751364844305763 0.012614402142041691 0.008413452467732944
0.005439745188247602 0.012325446525351304 0.00786979134951238
0.0052806082070531585 0.012902774240611996 0.007642289445314448
0.005083399660550398 0.01205056417507656 0.007053210466513354
0.004862419432474561 0.012851402499195735 0.00723528112236098
0.004575795870218318 0.012609487455627292 0.006820325713150256
0.003750515472141874 0.013233874276734411 0.005608587327787068
0.0030411312224328108 0.012410492138096639 0.004523612494013444
0.002481076444990523 0.013232662356040781 0.0038399901199035564
0.0019842349469126725 0.013428429092277469 0.0031229274068072972
0.0016835862382100484 0.013070015166110432 0.002457562505621736
0.0014703667635217015 0.0134972560476402 0.0020405399801483952
0.0013013145758497629 0.01440424013316709 0.0017574992535207862
0.0011301045738819288 0.015926163785044068 0.0014975396129370837
0.0010483592359552174 0.017583752205290636 0.0013513641245591736


In [50]:
# save to file
with open('rep.norm.Z.save','w') as file_:
    s = '# rep , norm^2, std(norm^2), Znorm^2, std(Znorm^2), Z, std(Z) '
    s += '\n'
    file_.write(s)
    for i_rep in range(n_rep):
        rep = rep_list[i_rep]
        s = '{:}'.format(rep)
        s += '  {:.16e}  {:.16e}'.format(norm_rep_z[i_rep],std_norm_z[i_rep])
        s += '  {:.16e}  {:.16e}'.format(Z_rep[i_rep]*norm_rep_z[i_rep],std_Znorm[i_rep])
        s += '  {:.16e}  {:.16e}'.format(Z_rep[i_rep],std_Z[i_rep])
        print(s)
        s += '\n'
        file_.write(s)

0  7.9980750000000045e-01  5.7513648443057626e-03  -7.1311750000000029e-01  8.4134524677329445e-03  -8.9161141899769625e-01  1.2614402142041691e-02
1  7.6591250000000000e-01  5.4397451882476017e-03  -6.8304250000000000e-01  7.8697913495123801e-03  -8.9180226200773594e-01  1.2325446525351304e-02
2  7.3504750000000019e-01  5.2806082070531585e-03  -6.5400625000000034e-01  7.6422894453144477e-03  -8.8974692111734299e-01  1.2902774240611996e-02
3  7.0481750000000021e-01  5.0833996605503980e-03  -6.2730125000000014e-01  7.0532104665133539e-03  -8.9001940218567210e-01  1.2050564175076561e-02
4  6.7457624999999988e-01  4.8624194324745611e-03  -6.0076125000000025e-01  7.2352811223609804e-03  -8.9057575033215342e-01  1.2851402499195735e-02
5  6.4657250000000066e-01  4.5757958702183181e-03  -5.7552125000000010e-01  6.8203257131502562e-03  -8.9011093110207984e-01  1.2609487455627292e-02
10  5.2146625000000046e-01  3.7505154721418740e-03  -4.6391750000000048e-01  5.6085873277870683e-03  -8.89640508

In [37]:
#i = 0 # only consider the ground state
##for i_rep in range(n_rep):
##for i_rep in range(5):
#for i_rep in range(7,n_rep):
#    rep = rep_list[i_rep]
#    # real part measurement
#    circuit_real = QuantumCircuit(qr,name='-')
#    Circuit_Initialize(circuit_real,i,qr)
#    circuit_real.h(qr[0])
#    for r in range(rep):
#        circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
#        circuit_real.cu(-params[0],-params[2],-params[1],-params[3],qr[0],qr[1])
#    circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
#    circuit_real.p(params[4],qr[0])
#    circuit_real.h(qr[0])
#
#    circuit_real = transpile(circuit_real,backend)
#
#    eps = eigen_vectors_exact[0,:,i].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i]
#    eps = eps.real
#    for alpha in range(1,n_hamiltonians):
#        params_list = []
#        # 0; norm measurement
#        i_obs = 0
#        for imc in range(nmc):
#            U_evol = np.identity(dim,dtype=complex)
#            times  = O_timelists[i][alpha][i_obs][imc]
#            phase  = 0.0
#            i_time = 0
#            # construction of P
#            for alpha_ in range(1,alpha):
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                phase += eigen_energies_rep[i_rep,alpha_] * time
#                i_time += 1
#
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha,time)@U_evol
#            phase += eps * time
#            i_time += 1
#
#            # construction of P^\dagger
#
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha,time)@U_evol
#            phase += eps * time
#            i_time += 1
#
#            for alpha_ in reversed(range(1,alpha)):
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                phase += eigen_energies_rep[i_rep,alpha_] * time
#                i_time += 1
#
#            theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#            params_list.append([theta,phi,lam,gamma,phase])
#        # 1; dE1 measurement
#        i_obs = 1
#        nhd1 = len(hamiltonian_diffs[alpha-1])
#        for ihd in range(nhd1):
#            # pass for constant contribution
#            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#                continue
#            for imc in range(nmc):
#                U_evol = np.identity(dim,dtype=complex)
#                times  = O_timelists[i][alpha][i_obs][imc]
#                phase  = 0.0
#                i_time = 0
#                # construction of P
#                for alpha_ in range(1,alpha):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#
#                # insertion of (H[alpha]-H[alpha-1])
#                Mat = hamiltonian_diffs[alpha-1].paulis[ihd].to_matrix()
#
#                U_evol = Mat@U_evol
#
#                # construction of P; continued
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                # construction of P^\dagger
#
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                for alpha_ in reversed(range(1,alpha)):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#                params_list.append([theta,phi,lam,gamma,phase])
#
#        # 2; dE2 measurement
#        i_obs = 2
#        if (alpha<n_hamiltonians-1):
#            nhd2 = len(hamiltonian_diffs[alpha])
#        else:
#            nhd2 = 0
#        for ihd in range(nhd2):
#            # pass for constant contribution
#            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#                continue
#            for imc in range(nmc):
#                U_evol = np.identity(dim,dtype=complex)
#                times  = O_timelists[i][alpha][i_obs][imc]
#                phase  = 0.0
#                i_time = 0
#                # construction of P
#                for alpha_ in range(1,alpha):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#
#                # construction of P; continued
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                # insertion of (H[alpha+1]-H[alpha])
#                Mat = hamiltonian_diffs[alpha].paulis[ihd].to_matrix()
#
#                U_evol = Mat@U_evol
#
#                # construction of P^\dagger
#
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                for alpha_ in reversed(range(1,alpha)):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#                params_list.append([theta,phi,lam,gamma,phase])
#        # 3; Z measurement
#        if (alpha==n_hamiltonians-1):
#            i_obs = 2
#            for imc in range(nmc):
#                U_evol = np.identity(dim,dtype=complex)
#                times  = O_timelists[i][alpha][i_obs][imc]
#                phase  = 0.0
#                i_time = 0
#                # construction of P
#                for alpha_ in range(1,alpha):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                # insertion of Z
#
#                U_evol = z_mat@U_evol
#
#                # construction of P^\dagger
#
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                for alpha_ in reversed(range(1,alpha)):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#
#                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#                params_list.append([theta,phi,lam,gamma,phase])
#
#        circuits = [circuit_real.assign_parameters({params: params_}) for params_ in params_list]
#
#        z_hadamards = [z_hadamard_test]*len(circuits)
#        estimator = AerEstimator(backend_options={
#                "method": "density_matrix",
#                "coupling_map": coupling_map,
#                "noise_model": noise_model,
#                "basis_gates": basis_gates
#            })
#        job = estimator.run(circuits,z_hadamards)
#        result_values = job.result().values
#        result_values_rep[i_rep][alpha] = result_values
#        # compute values
#        norm    = 0.0
#        dE1     = 0.0
#        dE2     = 0.0
#
#        i_meas = 0
#        # 0; norm
#        for imc in range(nmc):
#            norm += result_values_rep[i_rep][alpha][i_meas]
#            i_meas += 1
#        # 1; dE1
#        nhd1 = len(hamiltonian_diffs[alpha-1])
#        for ihd in range(nhd1):
#            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#                continue
#            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
#            for imc in range(nmc):
#                dE1 +=  coeff * result_values_rep[i_rep][alpha][i_meas]
#                i_meas += 1
#
#        # 2; dE2
#        if (alpha<n_hamiltonians-1):
#            nhd2 = len(hamiltonian_diffs[alpha])
#        else:
#            nhd2 = 0
#        for ihd in range(nhd2):
#            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#                continue
#            coeff = hamiltonian_diffs_list[alpha][ihd][1]
#            for imc in range(nmc):
#                dE2 += coeff * result_values_rep[i_rep][alpha][i_meas]
#                i_meas += 1
#
#        dE1     /=norm
#        dE2     /=norm
#
#        # 3; Z
#        if (alpha==n_hamiltonians-1):
#            Znorm = 0.0
#            for imc in range(nmc):
#                Znorm += result_values_rep[i_rep][alpha][i_meas]
#                i_meas += 1
#            Znorm /= norm
#
#        norm    /=nmc
#
#        # add constant contributions
#        for ihd in range(nhd1):
#            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]
#
#        for ihd in range(nhd2):
#            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#                dE2 += hamiltonian_diffs_list[alpha][ihd][1]
#
#        eigen_energies_rep[i_rep,alpha] = eigen_energies_rep[i_rep,alpha-1] + dE1
#        norms_rep[i_rep,alpha] = norm
#
#        if (alpha<n_hamiltonians-1):
#            eps = eigen_energies_rep[i_rep,alpha] + dE2
#            eps = eps.real
#        else:
#            Z_rep[i_rep] = Znorm
#
#        print(alpha, norms_rep[i_rep,alpha], eigen_energies_rep[i_rep,alpha]-eigen_energies_exact[alpha,i])
#        if (alpha<n_hamiltonians-1):
#            print(alpha, eps, eigen_energies_exact[alpha+1,i])
#        else:
#            print('# Zdiff: ', Znorm-z_exps_exact[-1,i])
#        st = '# {i_rep}/{n_rep}: {percent}%'.format(i_rep=i_rep+1,n_rep=n_rep,percent=((alpha)/(n_hamiltonians-1)*100))
#
#        del job
#        del z_hadamards
#        del circuits 
#        del estimator
#        collected = gc.collect()
#        memory_usage(st)
#
#
#

/tmp/ipykernel_40757/1109024003.py:256: ComplexWarning: Casting complex values to real discards the imaginary part
  eigen_energies_rep[i_rep,alpha] = eigen_energies_rep[i_rep,alpha-1] + dE1


1 0.485556640625 0.0005597158557776316
1 -0.7726427053887395 -0.7810249675906655
[# 8/15: 10.0%] memory usage:    0.78391 GiB
2 0.4862060546875 -0.00020258366330949684
2 -0.6277492685524937 -0.6403124237432848
[# 8/15: 20.0%] memory usage:    0.79208 GiB
3 0.4791259765625 -0.0008876115871233603
3 -0.5142456404259497 -0.5385164807134504
[# 8/15: 30.0%] memory usage:    0.81023 GiB
4 0.465703125 -0.0025095066384307474
4 -0.466449573998417 -0.5
[# 8/15: 40.0%] memory usage:    0.83272 GiB
5 0.440908203125 -0.003151394337639335
5 -0.49844032653990294 -0.5385164807134505
[# 8/15: 50.0%] memory usage:    0.82901 GiB
6 0.41859375 -0.0026434989172086087
6 -0.6173987814223725 -0.640312423743285
[# 8/15: 60.0%] memory usage:    0.85971 GiB
7 0.4214208984375 -0.0028253792388982513
7 -0.7727680763087963 -0.7810249675906655
[# 8/15: 70.0%] memory usage:    0.81748 GiB
8 0.4267626953125 -0.0007382515766871656
8 -0.9367889053723161 -0.9433981132056604
[# 8/15: 80.0%] memory usage:    0.83058 GiB
9 0.

In [38]:
#with open('MDH.rep.results','wb') as file_:
#    pickle.dump([result_values_rep],file_)
